In [1]:
import json
from torch.utils.data import Dataset, DataLoader
import torch
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from torch.utils.data import DataLoader, random_split

In [45]:
# 1. Load JSON data
with open("/content/classfiy_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

texts = [item["question"] for item in data]
labels = [item["classification"] for item in data]

In [46]:
# 2. Encode intents to integers
label_encoder = LabelEncoder()
labels_encoded = label_encoder.fit_transform(labels)
num_classes = len(label_encoder.classes_)

In [47]:
# 3. Load AraBERT tokenizer and model
model_name = "aubmindlab/bert-base-arabertv02"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [49]:
# 4. PyTorch Dataset
class classificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=32):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }

dataset = classificationDataset(texts, labels_encoded, tokenizer)
# Split dataset (80/20)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

In [52]:
# 5. Classifier Model
class Classifier(nn.Module):
    def __init__(self, bert_model, hidden_dim=15, num_classes=num_classes, dropout=0.3):
        super(Classifier, self).__init__()
        self.bert = bert_model
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(self.bert.config.hidden_size, hidden_dim)
        self.relu = nn.ReLU()
        self.out = nn.Linear(hidden_dim, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # CLS token
        x = self.dropout(cls_embedding)
        x = self.fc(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.out(x)
        return x

In [53]:

model = Classifier(bert_model)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Total parameters
total_params = sum(p.numel() for p in model.parameters())
# Trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

Total parameters: 135,204,911
Trainable parameters: 135,204,911


In [54]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report

epochs = 10

for epoch in range(epochs):
    model.train()
    train_losses = []

    loop = tqdm(train_loader, leave=False, desc=f"Epoch {epoch+1} Training")
    for batch in loop:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        loop.set_postfix(loss=loss.item())

    avg_train_loss = sum(train_losses) / len(train_losses)

    model.eval()
    val_losses = []
    all_preds = []
    all_labels = []

    val_loop = tqdm(val_loader, leave=False, desc=f"Epoch {epoch+1} Validation")
    with torch.no_grad():
        for batch in val_loop:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            val_losses.append(loss.item())

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            val_loop.set_postfix(loss=loss.item())

    avg_val_loss = sum(val_losses) / len(val_losses)
    val_acc = accuracy_score(all_labels, all_preds)
    class_report = classification_report(all_labels, all_preds, target_names=label_encoder.classes_)

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print("Classification Report:\n", class_report)

Epoch 1/10 | Train Loss: 0.3432 | Val Loss: 0.2538 | Val Acc: 0.8881
Classification Report:
               precision    recall  f1-score   support

     complex       0.87      0.92      0.90       850
      simple       0.90      0.85      0.88       750

    accuracy                           0.89      1600
   macro avg       0.89      0.89      0.89      1600
weighted avg       0.89      0.89      0.89      1600



Epoch 2/10 | Train Loss: 0.2374 | Val Loss: 0.2433 | Val Acc: 0.8894
Classification Report:
               precision    recall  f1-score   support

     complex       0.93      0.85      0.89       850
      simple       0.85      0.93      0.89       750

    accuracy                           0.89      1600
   macro avg       0.89      0.89      0.89      1600
weighted avg       0.89      0.89      0.89      1600



Epoch 3/10 | Train Loss: 0.2001 | Val Loss: 0.2646 | Val Acc: 0.8881
Classification Report:
               precision    recall  f1-score   support

     complex       0.96      0.83      0.89       850
      simple       0.83      0.96      0.89       750

    accuracy                           0.89      1600
   macro avg       0.89      0.89      0.89      1600
weighted avg       0.90      0.89      0.89      1600



Epoch 4/10 | Train Loss: 0.1620 | Val Loss: 0.2705 | Val Acc: 0.9094
Classification Report:
               precision    recall  f1-score   support

     complex       0.94      0.89      0.91       850
      simple       0.88      0.93      0.91       750

    accuracy                           0.91      1600
   macro avg       0.91      0.91      0.91      1600
weighted avg       0.91      0.91      0.91      1600



Epoch 5/10 | Train Loss: 0.1320 | Val Loss: 0.2951 | Val Acc: 0.9000
Classification Report:
               precision    recall  f1-score   support

     complex       0.91      0.90      0.91       850
      simple       0.89      0.89      0.89       750

    accuracy                           0.90      1600
   macro avg       0.90      0.90      0.90      1600
weighted avg       0.90      0.90      0.90      1600



Epoch 6/10 | Train Loss: 0.1132 | Val Loss: 0.3322 | Val Acc: 0.9000
Classification Report:
               precision    recall  f1-score   support

     complex       0.94      0.87      0.90       850
      simple       0.86      0.93      0.90       750

    accuracy                           0.90      1600
   macro avg       0.90      0.90      0.90      1600
weighted avg       0.90      0.90      0.90      1600



Epoch 7/10 | Train Loss: 0.0875 | Val Loss: 0.3224 | Val Acc: 0.9056
Classification Report:
               precision    recall  f1-score   support

     complex       0.92      0.91      0.91       850
      simple       0.89      0.91      0.90       750

    accuracy                           0.91      1600
   macro avg       0.91      0.91      0.91      1600
weighted avg       0.91      0.91      0.91      1600



Epoch 8/10 | Train Loss: 0.0802 | Val Loss: 0.3659 | Val Acc: 0.8962
Classification Report:
               precision    recall  f1-score   support

     complex       0.92      0.88      0.90       850
      simple       0.87      0.91      0.89       750

    accuracy                           0.90      1600
   macro avg       0.90      0.90      0.90      1600
weighted avg       0.90      0.90      0.90      1600



Epoch 9/10 | Train Loss: 0.0684 | Val Loss: 0.4103 | Val Acc: 0.8994
Classification Report:
               precision    recall  f1-score   support

     complex       0.94      0.87      0.90       850
      simple       0.86      0.93      0.90       750

    accuracy                           0.90      1600
   macro avg       0.90      0.90      0.90      1600
weighted avg       0.90      0.90      0.90      1600



Epoch 10/10 | Train Loss: 0.0570 | Val Loss: 0.3834 | Val Acc: 0.8981
Classification Report:
               precision    recall  f1-score   support

     complex       0.92      0.88      0.90       850
      simple       0.87      0.92      0.89       750

    accuracy                           0.90      1600
   macro avg       0.90      0.90      0.90      1600
weighted avg       0.90      0.90      0.90      1600



In [58]:
import torch.nn.functional as F

model.eval()
example_text = "عايز مقارنة بين لاب ديل واتش بى من حيث السعر والطاقة والشحن"
# Tokenize
encoding = tokenizer(
    example_text,
    return_tensors="pt",
    padding="max_length",
    truncation=True,
    max_length=32
)

# Move tensors to the same device as model
input_ids = encoding['input_ids'].to(device)
attention_mask = encoding['attention_mask'].to(device)

# Predict
with torch.no_grad():
    logits = model(input_ids, attention_mask)
    probs = F.softmax(logits, dim=1)  # Convert logits to probabilities
    max_prob, pred_class_tensor = torch.max(probs, dim=1)
    max_prob = max_prob.item()
    pred_class = pred_class_tensor.item()

    if max_prob < 0.8:  # threshold for unknown intent
        pred_intent = "not know"
    else:
        pred_intent = label_encoder.inverse_transform([pred_class])[0]

print("Predicted intent:", pred_intent)
print("Confidence:", max_prob)

Predicted intent: complex
Confidence: 0.9986264705657959


In [59]:
import zipfile
import os
import pickle

# Paths
model_path = "arabic_model.pt"
tokenizer_path = "arabic_tokenizer"
classes_path = "label_classes.pkl"

# Make sure tokenizer folder exists
if not os.path.exists(tokenizer_path):
    os.makedirs(tokenizer_path)
tokenizer.save_pretrained(tokenizer_path)

# Save model
torch.save(model.state_dict(), model_path)

# Save label encoder classes
with open(classes_path, "wb") as f:
    pickle.dump(label_encoder.classes_, f)

# Create a ZIP file
zip_filename = "arabic_model_all.zip"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Add model weights
    zipf.write(model_path)

    # Add label classes
    zipf.write(classes_path)

    # Add all tokenizer files
    for root, dirs, files in os.walk(tokenizer_path):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.join("tokenizer", file)  # keep inside a folder in ZIP
            zipf.write(file_path, arcname)

print(f"All files saved and zipped into {zip_filename}")

All files saved and zipped into arabic_model_all.zip
